# NB0 — Baseline "Prior" (thiên vị đa số / majority-class)

**Vai trò:** đây là *sàn tham chiếu* (floor) cho toàn bộ thực nghiệm. Mô hình không
học gì từ ảnh: với mỗi câu hỏi nó luôn đoán **đáp án phổ biến nhất trong tập train**.

Trong tài liệu VQA, baseline này còn gọi là **"language prior"** — cho thấy mức
điểm mà một mô hình có thể đạt được *chỉ nhờ đoán mù theo phân phối đáp án*, tức là
mức "thiên vị ngôn ngữ" của bộ dữ liệu. Mọi mô hình học thật (NB1, NB2) phải vượt
qua mốc này thì mới chứng minh được là ảnh/đặc trưng thực sự có ích.

Tham chiếu: Goyal et al. 2017 (VQA v2, khử language-prior); "Loss re-scaling VQA"
(arXiv 2010.16010) diễn giải prior như bài toán mất cân bằng lớp.

In [1]:
import os, re, sys, io
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score

QA = os.path.join(os.path.dirname(os.getcwd()), "qa") if os.path.basename(os.getcwd())=="experiments" else "F:/VQA/qa"
QID_ORDER = ["Q1","Q2","Q3","Q4","Q5","Q6","Q7","Q8","Q9","Q10"]
WHAT = {"Q2","Q5","Q8"}   # câu đa nhãn (multi-label)
SEED = 42

def norm(s): return re.sub(r"\s+"," ",str(s).strip().lower())

full = pd.concat([pd.read_csv(os.path.join(QA,f"{s}.csv")) for s in ["train","val","test"]], ignore_index=True)
full["qid"] = full["qid"].astype(str)
train = pd.read_csv(os.path.join(QA,"train.csv")); train["qid"]=train["qid"].astype(str)
test  = pd.read_csv(os.path.join(QA,"test.csv"));  test["qid"]=test["qid"].astype(str)
print("Tổng:", len(full), "dòng /", full["filename"].nunique(), "ảnh")

Tổng: 19474 dòng / 6349 ảnh


## 1. EDA — đáp án đa số của mỗi câu

Prior chỉ cần biết đáp án nào chiếm đa số trong train. Bảng dưới cho thấy đáp án đó
và tỉ lệ của nó — cũng chính là *accuracy mà prior sẽ đạt* nếu test có cùng phân phối.

In [2]:
rows=[]
for q in QID_ORDER:
    sub = train[train["qid"]==q]
    vc = sub["answer"].map(norm).value_counts()
    rows.append({"qid":q, "n_train":len(sub), "đáp_án_đa_số":vc.index[0],
                 "tỉ_lệ":round(vc.iloc[0]/len(sub),3)})
pd.DataFrame(rows)

,qid,n_train,đáp_án_đa_số,tỉ_lệ
0,Q1,3137,có,0.730
1,Q2,2048,không có,0.582
2,Q3,1567,không,0.610
3,Q4,211,không,0.782
4,Q5,1830,xe máy,0.492
5,Q6,1487,0,0.356
6,Q7,770,không,0.883
7,Q8,1094,không có,0.431
8,Q9,559,không,0.796
9,Q10,969,có,0.834


## 2. Đánh giá prior trên tập test (single-split)

Với mỗi câu, đoán đáp án đa số của train cho *mọi* ảnh test rồi đo EM (accuracy) và
F1-macro. Câu đa nhãn (Q2/Q5/Q8): prior = tập nhãn đa số (thường là "không có").

In [3]:
def prior_eval(qids, df_tr, df_te):
    out=[]
    for q in qids:
        maj = df_tr[df_tr["qid"]==q]["answer"].map(norm).value_counts().index[0]
        yte = df_te[df_te["qid"]==q]["answer"].map(norm).values
        yp  = np.array([maj]*len(yte), object)
        out.append({"qid":q,"n_test":len(yte),
                    "EM":round(accuracy_score(yte,yp),4),
                    "F1_macro":round(f1_score(yte,yp,average="macro",zero_division=0),4),
                    "R_macro":round(recall_score(yte,yp,average="macro",zero_division=0),4)})
    return pd.DataFrame(out)

tbl = prior_eval(QID_ORDER, train, test)
tbl.loc["TB"] = ["—","—",round(tbl["EM"].mean(),4),round(tbl["F1_macro"].mean(),4),round(tbl["R_macro"].mean(),4)]
tbl

,qid,n_test,EM,F1_macro,R_macro
0,Q1,898,0.7183,0.4180,0.5000
1,Q2,577,0.5945,0.0746,0.1000
2,Q3,412,0.5218,0.3429,0.5000
3,Q4,66,0.8485,0.4590,0.5000
4,Q5,559,0.4794,0.1080,0.1667
5,Q6,399,0.3584,0.0879,0.1667
6,Q7,209,0.9139,0.4775,0.5000
7,Q8,324,0.4383,0.0406,0.0667
8,Q9,179,0.7933,0.4424,0.5000
9,Q10,268,0.8619,0.4629,0.5000


## 3. Kiểm chứng bằng 5-fold CV (đáng tin hơn single-split)

Test nhỏ nên single-split nhiễu. Ta lặp prior trên 5-fold: mỗi fold lấy đáp án đa số
từ phần train của fold rồi đo trên phần test của fold. Báo cáo trung bình ± độ lệch.

In [4]:
def prior_cv(qids, folds=5):
    out=[]
    for q in qids:
        sub = full[full["qid"]==q].reset_index(drop=True)
        y = sub["answer"].map(norm).values
        if len(set(y))<2:
            out.append({"qid":q,"EM_mean":round((y==pd.Series(y).mode()[0]).mean(),4),"EM_std":0.0,
                        "F1macro_mean":0.0,"F1macro_std":0.0}); continue
        skf=StratifiedKFold(n_splits=folds,shuffle=True,random_state=SEED)
        ems,f1s=[],[]
        for tr,te in skf.split(np.zeros(len(y)),y):
            maj=pd.Series(y[tr]).value_counts().index[0]
            yp=np.array([maj]*len(te),object)
            ems.append(accuracy_score(y[te],yp)); f1s.append(f1_score(y[te],yp,average="macro",zero_division=0))
        out.append({"qid":q,"EM_mean":round(np.mean(ems),4),"EM_std":round(np.std(ems),4),
                    "F1macro_mean":round(np.mean(f1s),4),"F1macro_std":round(np.std(f1s),4)})
    return pd.DataFrame(out)

cv = prior_cv(QID_ORDER)
print("Prior 5-fold CV — trung bình EM   =", round(cv["EM_mean"].mean(),4))
print("Prior 5-fold CV — trung bình F1ma =", round(cv["F1macro_mean"].mean(),4))
cv

C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Prior 5-fold CV — trung bình EM   = 0.6504
Prior 5-fold CV — trung bình F1ma = 0.2921


,qid,EM_mean,EM_std,F1macro_mean,F1macro_std
0,Q1,0.7281,0.0003,0.4213,0.0001
1,Q2,0.5853,0.0011,0.0788,0.0041
2,Q3,0.5896,0.0008,0.3709,0.0003
3,Q4,0.8121,0.0052,0.4482,0.0016
4,Q5,0.4910,0.0013,0.1098,0.0002
5,Q6,0.3513,0.0006,0.0867,0.0001
6,Q7,0.8855,0.0003,0.4696,0.0001
7,Q8,0.4317,0.0007,0.0377,0.0000
8,Q9,0.7888,0.0027,0.4410,0.0008
9,Q10,0.8408,0.0003,0.4568,0.0001


## 4. Kết luận NB0

- Prior đạt **EM cao** ở các câu lệch mạnh (Q1, Q7...) chỉ nhờ đoán đáp án đa số,
  nhưng **F1-macro rất thấp** vì hoàn toàn bỏ lớp thiểu số → đúng bản chất bài toán
  mất cân bằng.
- Đây là mốc sàn: NB1 (ViT+MLP) và NB2 (PhoBERT+ViT) phải vượt **F1-macro** này để
  chứng minh giá trị của đặc trưng ảnh và của các phương pháp cân bằng.

Lưu bảng kết quả để dùng cho báo cáo Chương 5.

In [5]:
os.makedirs("results", exist_ok=True)
tbl.to_csv("results/nb0_prior_split.csv", index=False)
cv.to_csv("results/nb0_prior_cv.csv", index=False)
print("Đã lưu results/nb0_prior_split.csv và results/nb0_prior_cv.csv")

Đã lưu results/nb0_prior_split.csv và results/nb0_prior_cv.csv
